In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

#Cargar datos

In [3]:
df = pd.read_csv(
    "Telco_customer_churn.csv",
    sep=";"
)

print(df.shape)

(7043, 33)


#Seleccionar variables

In [4]:
columnas = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Label"
]

df_ml = df[columnas].copy()
print(df_ml.shape)

(7043, 4)


#Convertir Churn a valores numéricos

In [5]:
print(df_ml.shape)

df_ml["Churn Label"].value_counts()

(7043, 4)


,count
Churn Label,
No,5174
Yes,1869


In [6]:
df_ml["Churn Label"] = df_ml["Churn Label"].map({
    "Yes": 1,
    "No": 0
})

print(df_ml.shape)

df_ml["Churn Label"].value_counts(dropna=False)

(7043, 4)


,count
Churn Label,
0,5174
1,1869


In [7]:
df_ml["Total Charges"].dtype

dtype('O')

In [8]:
df_ml["Total Charges"].head(10)

,Total Charges
0,108.15
1,151.65
2,820.5
3,3046.05
4,5036.3
5,528.35
6,39.65
7,20.15
8,4749.15
9,30.2


In [9]:
df_ml["Total Charges"] = pd.to_numeric(
    df_ml["Total Charges"],
    errors="coerce"
)

In [10]:
df_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Tenure Months    7043 non-null   int64  
 1   Monthly Charges  7043 non-null   float64
 2   Total Charges    7032 non-null   float64
 3   Churn Label      7043 non-null   int64  
dtypes: float64(2), int64(2)
memory usage: 220.2 KB


In [11]:
df_ml.isnull().sum()

,0
Tenure Months,0
Monthly Charges,0
Total Charges,11
Churn Label,0


In [12]:
df_ml = df_ml.dropna()

print(df_ml.shape)

(7032, 4)


In [13]:
X = df_ml[
    [
        "Tenure Months",
        "Monthly Charges",
        "Total Charges"
    ]
]

y = df_ml["Churn Label"]

In [14]:
print(X.shape)
print(y.shape)

(7032, 3)
(7032,)


#Dividir entrenamiento y prueba

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Datos divididos correctamente.")
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Datos divididos correctamente.
Entrenamiento: (5625, 3)
Prueba: (1407, 3)


In [16]:
print(X_train.shape)
print(X_test.shape)

(5625, 3)
(1407, 3)


#Entrenar el modelo

In [17]:
modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced"
)

modelo.fit(X_train, y_train)

print("Modelo Random Forest entrenado correctamente.")

Modelo Random Forest entrenado correctamente.


#Realizar predicciones

In [18]:
y_pred = modelo.predict(X_test)

# Probabilidad de que el cliente abandone
y_prob = modelo.predict_proba(X_test)[:, 1]

print("Predicciones generadas correctamente.")

Predicciones generadas correctamente.


#Obtener métrica

In [19]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

print("EVALUACIÓN DEL MODELO RANDOM FOREST")
print("-----------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

EVALUACIÓN DEL MODELO RANDOM FOREST
-----------------------------------
Accuracy:  0.7520
Precision: 0.5410
Recall:    0.4412
F1 Score:  0.4860
ROC-AUC:   0.7663


In [20]:
print("REPORTE DE CLASIFICACIÓN")
print("------------------------")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No abandona", "Abandona"],
        zero_division=0
    )
)

print("MATRIZ DE CONFUSIÓN")
print("-------------------")
print(confusion_matrix(y_test, y_pred))

REPORTE DE CLASIFICACIÓN
------------------------
              precision    recall  f1-score   support

 No abandona       0.81      0.86      0.84      1033
    Abandona       0.54      0.44      0.49       374

    accuracy                           0.75      1407
   macro avg       0.68      0.65      0.66      1407
weighted avg       0.74      0.75      0.74      1407

MATRIZ DE CONFUSIÓN
-------------------
[[893 140]
 [209 165]]


## Conclusiones

Se entrenó un modelo de clasificación utilizando el algoritmo
Random Forest para predecir el abandono de clientes (Customer Churn).

Para evaluar su rendimiento se utilizaron las métricas Accuracy,
Precision, Recall, F1 Score y ROC-AUC.

Los resultados obtenidos fueron:

- Accuracy: 75.20%
- Precision: 54.10%
- Recall: 44.12%
- F1 Score: 48.60%
- ROC-AUC: 76.63%

Los resultados muestran que el modelo alcanza una capacidad de clasificación general del 75.20%. Sin embargo, el Recall de 44.12% indica que todavía presenta dificultades para identificar a todos los clientes que realmente abandonan el servicio.

La matriz de confusión registró 893 verdaderos negativos, 140 falsos positivos, 209 falsos negativos y 165 verdaderos positivos.

Esto significa que el modelo identificó correctamente 165 clientes que abandonaron, pero no detectó otros 209 casos reales de abandono. Por ello, será importante considerar el Recall como una métrica relevante al evaluar su utilidad para la detección temprana de clientes en riesgo.

Finalmente, el modelo entrenado fue guardado en formato PKL para permitir su reutilización en las siguientes etapas del proyecto.

Estos resultados servirán como referencia para comparar el modelo clásico Random Forest con la red neuronal desarrollada en US-06 y seleccionar el modelo más adecuado para el sistema predictivo de abandono de clientes.

In [21]:
import os

# Crear la carpeta models si todavía no existe
os.makedirs("models", exist_ok=True)

# Guardar el modelo entrenado
ruta_modelo = "models/modelo_random_forest_churn.pkl"

joblib.dump(modelo, ruta_modelo)

print(f"Modelo guardado correctamente en: {ruta_modelo}")

Modelo guardado correctamente en: models/modelo_random_forest_churn.pkl
